In [ ]:
import re

def parse_distance_changes(filename):
    distance_list = []  
    residue_set = set()

    with open(filename, 'r') as f:
        lines = f.readlines()[2:]
        for line in lines:
            if not line.strip():
                continue
            parts = line.strip().split('\t')
            if len(parts) < 3:
                continue
            res1, res2, diff = parts[0], parts[1], float(parts[2])

            def normalize_res(r):
                if r.startswith('A_') and r[2:5] == 'HIS':
                    return 'A_HSD' + r[5:]
                return r

            res1 = normalize_res(res1)
            res2 = normalize_res(res2)

            distance_list.append((res1, res2))

            if 'centroid' not in res1:
                residue_set.add(res1)
            if 'centroid' not in res2:
                residue_set.add(res2)

    return distance_list, sorted(residue_set)

def parse_dihedral_changes(filename):
    dihedral_list = []
    with open(filename, 'r') as f:
        lines = f.readlines()[2:]
        for line in lines:
            if not line.strip():
                continue
            parts = line.strip().split('\t')
            if len(parts) < 3:
                continue
            residue, angle_type, diff = parts[0], parts[1], float(parts[2])

            key = (residue[2:], angle_type)
            dihedral_list.append(key)
            
    return dihedral_list

def get_residue_atoms(pdb_file):
    atoms_dict = {}
    with open(pdb_file, 'r') as f:
        for line in f:
            if not (line.startswith('ATOM  ') or line.startswith('HETATM')):
                continue
            try:
                chain_id = line[21]
                atom_num = int(line[6:11])
                res_name = line[17:20].strip()
                res_num = int(line[22:26])
            except Exception:
                continue
            res_id = f"{chain_id}_{res_name}{res_num}"
            atoms_dict.setdefault(res_id, []).append(atom_num)
    return atoms_dict

def get_ca_atoms(pdb_file, chain, start, end):
    ca_list = []
    with open(pdb_file, 'r') as f:
        for line in f:
            if not (line.startswith('ATOM  ') or line.startswith('HETATM')):
                continue
            try:
                chain_id = line[21]
                atom_name = line[12:16].strip()
                atom_num = int(line[6:11])
                res_num = int(line[22:26])
            except Exception:
                continue
            if chain_id != chain:
                continue
            if atom_name != 'CA':
                continue
            if start <= res_num <= end:
                ca_list.append(atom_num)
    return [str(x) for x in sorted(set(ca_list))]

def write_plumed_input(distance_list, dihedral_list, molinfo_pdb, centroid_pdb=None):
    atoms_dict = get_residue_atoms(molinfo_pdb)
    centroid_pdb = centroid_pdb if centroid_pdb else molinfo_pdb

    centers = {}

    with open('plumed_from_2structures.dat', 'w') as f:
        f.write(f'MOLINFO STRUCTURE={molinfo_pdb}\n')

        for res1, res2 in distance_list:
            for res in (res1, res2):
                if 'centroid' not in res:
                    continue
                parts = res.split('_')
                if len(parts) < 3:
                    continue
                ch = parts[0]
                rng = parts[-1]
                key = f"{ch}_{rng}"
                if key in centers:
                    continue
                try:
                    start, end = map(int, rng.split('-'))
                except Exception:
                    continue
                ca_atoms = get_ca_atoms(centroid_pdb, ch, start, end)
                if not ca_atoms:
                    print(f"Warning: no CA atoms for centroid {res} in {centroid_pdb}")
                    continue
                center_label = f"{ch}_center_{start}_{end}"
                f.write(f"{center_label}: CENTER ATOMS={','.join(ca_atoms)}\n")
                centers[key] = center_label

        for res1, res2 in distance_list:
            res1_is_centroid = 'centroid' in res1
            res2_is_centroid = 'centroid' in res2

            if res1_is_centroid:
                parts = res1.split('_')
                ch = parts[0]
                rng = parts[-1]
                key = f"{ch}_{rng}"
                groupA = centers.get(key)
                groupA_is_label = True if groupA else False
            else:
                atomsA = atoms_dict.get(res1, [])
                if not atomsA:
                    print(f"Warning: missing atoms for {res1} in {molinfo_pdb}")
                    continue
                groupA = ','.join([str(x) for x in atomsA])
                groupA_is_label = False

            if res2_is_centroid:
                parts = res2.split('_')
                ch = parts[0]
                rng = parts[-1]
                key = f"{ch}_{rng}"
                groupB = centers.get(key)
                groupB_is_label = True if groupB else False
            else:
                atomsB = atoms_dict.get(res2, [])
                if not atomsB:
                    print(f"Warning: missing atoms for {res2} in {molinfo_pdb}")
                    continue
                groupB = ','.join([str(x) for x in atomsB])
                groupB_is_label = False

            safe1 = re.sub(r'[^0-9A-Za-z]', '_', res1)
            safe2 = re.sub(r'[^0-9A-Za-z]', '_', res2)
            if groupA_is_label:
                f.write(f"d_{safe2}_to_{groupA}: DISTANCES GROUPA={groupB} GROUPB={groupA} MIN={{BETA=10.}}\n")
            elif groupB_is_label:
                f.write(f"d_{safe1}_to_{groupB}: DISTANCES GROUPA={groupA} GROUPB={groupB} MIN={{BETA=10.}}\n")
            else:
                f.write(f"d_{safe1}_{safe2}: DISTANCES GROUPA={groupA} GROUPB={groupB} MIN={{BETA=10.}}\n")
        for res, angle_type in dihedral_list:
            safe = re.sub(r'[^0-9A-Za-z]', '_', res)
            f.write(f"A_{angle_type}_{safe}: TORSION ATOMS=@{angle_type}-A{res[3:]}\n")

        f.write('PRINT ARG=* STRIDE=1 FILE=COLVAR\n')


In [ ]:
cross_file = 'significant_cross_changes.txt'
centroid_file = 'significant_ca_centroid_changes.txt'

distances_cross, residues_cross = parse_distance_changes(cross_file)
distances_centroid, residues_centroid = parse_distance_changes(centroid_file)

combined_distances = distances_cross + distances_centroid

dihedral_list = parse_dihedral_changes('significant_dihedral_changes.txt')

molinfo = 'open.pdb'
write_plumed_input(combined_distances, dihedral_list, molinfo_pdb=molinfo, centroid_pdb=molinfo)